In [ ]:
%%capture

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
!nvidia-smi

Sat Jun 20 15:17:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model.load_adapter(
    "IndusYash/investo-llama-3.2-3b-finance-lora"
)

FastLanguageModel.for_inference(model)

print("Investo Bot loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


adapter_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Investo Bot loaded successfully!


In [ ]:
def ask_investo_bot(question):
    messages = [
        {
            "role": "system",
            "content": (
                "You are Investo Bot, an expert AI financial assistant. "
                "Provide clear, accurate, and well-structured financial explanations. "
                "When discussing investments, explain risks and avoid unrealistic guarantees."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )

    print("\nInvesto Bot:")
    print(response)

In [ ]:
ask_investo_bot(
    "Explain the difference between a stock and a bond."
)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Investo Bot:
A stock is a type of investment that represents ownership in a company. When you buy a stock, you essentially become a shareholder of the company and are entitled to a portion of its profits. The value of the stock can fluctuate based on the company's performance, so you may receive more or less than the initial price paid.

A bond, on the other hand, is a type of debt investment. When you buy a bond, you essentially lend money to the borrower (the company or government) in exchange for regular interest payments and the return of your principal investment at maturity. Bonds typically offer a fixed interest rate and a fixed maturity date, so you know exactly how much you'll receive at the end of the bond's term.


In [ ]:
ask_investo_bot(
    "I want to put all my life savings into one stock because my friend said it will definitely double in one year. Should I do it?"
)

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Investo Bot:
No, it is not advisable to put all your life savings into one stock based on the recommendation of a friend. There are several risks associated with investing in the stock market, including:

1. Market volatility: The stock market can be unpredictable, and the value of your investment can fluctuate significantly over time.
2. Risk of loss: There is always a risk that your investment may lose value or even go to zero.
3. Lack of diversification: Putting all your eggs in one basket can be risky, as a decline in one stock can have a significant impact on your overall portfolio.
4. Lack of research: It is important to do your own research and due diligence before investing in any stock, as there may be risks and downsides to the investment that you are not aware of.

Instead of putting all your life savings into one stock, consider the following:

1. Diversify your portfolio: Spread your investments across a variety of stocks, bonds, and other assets to reduce risk.
2. Resear